# GLD Trading Strategy Backtesting

This notebook implements and backtests multiple trading strategies using ARIMA/LSTM price predictions:
1. **Optimal Multi-Signal Strategy** - Uses prediction confidence, momentum, and mean reversion
2. **Simple If/Elif/Else Strategy** - Basic threshold-based trading
3. **Comprehensive Backtesting** - Performance comparison on real GLD data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading and Alignment

In [ ]:
# Load GLD daily data
gld_data = pd.read_csv('../data/gld_daily.csv', parse_dates=['date'], index_col='date')
print(f"GLD data: {gld_data.shape[0]} days from {gld_data.index[0]} to {gld_data.index[-1]}")

# Load ARIMA/LSTM predictions from pricepredictor.ipynb
predictions = np.load('../data/final_predictions.npy')
print(f"Predictions: {len(predictions)} values, range: ${predictions.min():.2f} - ${predictions.max():.2f}")

# Load combined data to get prediction dates
combined_data = pd.read_csv('../data/combined_data.csv', parse_dates=['date'], index_col='date')
print(f"Combined data: {combined_data.shape[0]} days")

In [ ]:
# Align predictions with actual dates (based on pricepredictor.ipynb logic)
# The model used 40% for LSTM training and 40% for final test (176 predictions)
# Predictions start from the final test period (after 80% of data)

total_data = len(combined_data)
test_start_idx = int(0.8 * total_data)  # 80% for training (ARIMA + LSTM train)
test_end_idx = test_start_idx + len(predictions) + 14  # +14 for LSTM window

# Get the actual test period dates
test_dates = combined_data.index[test_start_idx + 14:test_start_idx + 14 + len(predictions)]

print(f"Test period: {test_dates[0]} to {test_dates[-1]}")
print(f"Predictions aligned with {len(test_dates)} trading days")

In [ ]:
# Create aligned dataset for backtesting
backtest_data = pd.DataFrame({
    'actual_price': gld_data.loc[test_dates, 'Close'],
    'predicted_price': predictions,
    'open_price': gld_data.loc[test_dates, 'Open'],
    'high': gld_data.loc[test_dates, 'High'],
    'low': gld_data.loc[test_dates, 'Low'],
    'volume': gld_data.loc[test_dates, 'Volume']
}, index=test_dates)

# Add technical indicators from original data
for col in ['EMA_30', 'EMA_60', 'EMA_200', 'rsi_14', 'MACD', 'BB_Upper', 'BB_Lower']:
    if col in gld_data.columns:
        backtest_data[col] = gld_data.loc[test_dates, col]

print(f"Backtest dataset: {backtest_data.shape}")
backtest_data.head()

## 2. Strategy Implementation

### 2.1 Simple If/Elif/Else Strategy

In [ ]:
def simple_strategy(data, buy_threshold=0.02, sell_threshold=-0.02):
    """
    Simple threshold-based strategy using prediction vs current price
    
    Args:
        data: DataFrame with 'actual_price' and 'predicted_price'
        buy_threshold: Minimum price change % to trigger BUY
        sell_threshold: Maximum price change % to trigger SELL
    
    Returns:
        Series of trading signals: 1=BUY, 0=HOLD, -1=SELL
    """
    
    signals = []
    
    for i in range(len(data)):
        current_price = data['actual_price'].iloc[i]
        predicted_price = data['predicted_price'].iloc[i]
        
        # Calculate expected return
        price_change_pct = (predicted_price - current_price) / current_price
        
        # Simple if/elif/else logic
        if price_change_pct >= buy_threshold:
            signal = 1  # BUY
        elif price_change_pct <= sell_threshold:
            signal = -1  # SELL
        else:
            signal = 0  # HOLD
            
        signals.append(signal)
    
    return pd.Series(signals, index=data.index, name='simple_signal')

# Generate signals
backtest_data['simple_signal'] = simple_strategy(backtest_data)

# Show signal distribution
print("Simple Strategy Signal Distribution:")
print(backtest_data['simple_signal'].value_counts().sort_index())
print(f"\nBuy signals: {(backtest_data['simple_signal'] == 1).sum()}")
print(f"Sell signals: {(backtest_data['simple_signal'] == -1).sum()}")
print(f"Hold signals: {(backtest_data['simple_signal'] == 0).sum()}")

### 2.2 Optimal Multi-Signal Strategy

In [ ]:
def optimal_strategy(data, lookback=10, confidence_threshold=1.2):
    """
    Advanced multi-signal strategy combining:
    1. Prediction vs current price (trend signal)
    2. Prediction confidence based on volatility
    3. Mean reversion component
    4. Momentum from prediction changes
    
    Args:
        data: DataFrame with prices and technical indicators
        lookback: Period for rolling calculations
        confidence_threshold: Minimum confidence for trades
    """
    
    df = data.copy()
    
    # 1. Basic prediction signal
    df['price_change_pct'] = (df['predicted_price'] - df['actual_price']) / df['actual_price']
    
    # 2. Prediction confidence (inverse of recent volatility)
    df['price_volatility'] = df['actual_price'].rolling(lookback).std() / df['actual_price'].rolling(lookback).mean()
    df['pred_volatility'] = df['predicted_price'].rolling(lookback).std() / df['predicted_price'].rolling(lookback).mean()
    df['confidence'] = 1 / (df['price_volatility'] + df['pred_volatility'] + 0.01)  # Add small constant to avoid division by zero
    
    # 3. Mean reversion signal
    df['pred_ma'] = df['predicted_price'].rolling(lookback).mean()
    df['mean_reversion'] = (df['predicted_price'] - df['pred_ma']) / df['pred_ma']
    
    # 4. Momentum signal (rate of change in predictions)
    df['pred_momentum'] = df['predicted_price'].pct_change(3).rolling(3).mean()  # 3-day momentum
    
    # 5. Technical indicator signals (if available)
    rsi_signal = 0
    if 'rsi_14' in df.columns:
        # RSI oversold/overbought signals
        rsi_signal = np.where(df['rsi_14'] < 30, 0.5,  # Oversold -> buy bias
                             np.where(df['rsi_14'] > 70, -0.5, 0))  # Overbought -> sell bias
    
    # Combine signals with weights
    signals = []
    
    for i in range(len(df)):
        if i < lookback:  # Not enough data for rolling calculations
            signals.append(0)
            continue
            
        # Get current values
        price_signal = df['price_change_pct'].iloc[i]
        confidence = df['confidence'].iloc[i]
        mean_rev = df['mean_reversion'].iloc[i]
        momentum = df['pred_momentum'].iloc[i] if not pd.isna(df['pred_momentum'].iloc[i]) else 0
        rsi = rsi_signal[i] if isinstance(rsi_signal, np.ndarray) else 0
        
        # Combined signal with weights
        combined_signal = (
            0.4 * price_signal +      # Primary: prediction vs price
            0.2 * momentum +          # Momentum component
            0.1 * (-mean_rev) +       # Mean reversion (negative because we want to trade against extremes)
            0.3 * rsi                 # Technical indicator
        )
        
        # Apply confidence filter and generate signal
        if confidence >= confidence_threshold:
            if combined_signal > 0.015:  # Strong buy
                signal = 1
            elif combined_signal < -0.015:  # Strong sell
                signal = -1
            else:
                signal = 0  # Hold
        else:
            signal = 0  # Low confidence -> hold
        
        signals.append(signal)
    
    return pd.Series(signals, index=df.index, name='optimal_signal')

# Generate optimal signals
backtest_data['optimal_signal'] = optimal_strategy(backtest_data)

print("Optimal Strategy Signal Distribution:")
print(backtest_data['optimal_signal'].value_counts().sort_index())
print(f"\nBuy signals: {(backtest_data['optimal_signal'] == 1).sum()}")
print(f"Sell signals: {(backtest_data['optimal_signal'] == -1).sum()}")
print(f"Hold signals: {(backtest_data['optimal_signal'] == 0).sum()}")

## 3. Backtesting Framework

In [ ]:
def backtest_strategy(data, signals, initial_capital=100000, transaction_cost=0.001):
    """
    Backtest a trading strategy with realistic assumptions
    
    Args:
        data: DataFrame with OHLCV data
        signals: Series of trading signals (1=buy, -1=sell, 0=hold)
        initial_capital: Starting capital in USD
        transaction_cost: Cost per transaction as % of trade value
    
    Returns:
        Dictionary with performance metrics and trade history
    """
    
    results = {
        'dates': [],
        'positions': [],  # 1=long, 0=cash, -1=short
        'shares': [],
        'cash': [],
        'portfolio_value': [],
        'trades': [],
        'returns': []
    }
    
    cash = initial_capital
    shares = 0
    position = 0  # 0=cash, 1=long, -1=short
    
    for i, (date, signal) in enumerate(signals.items()):
        current_price = data.loc[date, 'actual_price']
        
        # Trading logic
        trade_executed = False
        
        if signal == 1 and position <= 0:  # Buy signal
            if position == -1:  # Cover short first
                cash -= shares * current_price * (1 + transaction_cost)
                shares = 0
            
            # Buy long
            if cash > 0:
                shares_to_buy = int(cash / (current_price * (1 + transaction_cost)))
                if shares_to_buy > 0:
                    cost = shares_to_buy * current_price * (1 + transaction_cost)
                    cash -= cost
                    shares = shares_to_buy
                    position = 1
                    trade_executed = True
                    results['trades'].append((date, 'BUY', shares_to_buy, current_price))
        
        elif signal == -1 and position >= 0:  # Sell signal
            if position == 1:  # Sell long first
                cash += shares * current_price * (1 - transaction_cost)
                shares = 0
            
            # Short sell (simplified - assume we can short)
            shares_to_short = int(initial_capital * 0.5 / current_price)  # Limited short position
            if shares_to_short > 0:
                cash += shares_to_short * current_price * (1 - transaction_cost)
                shares = -shares_to_short
                position = -1
                trade_executed = True
                results['trades'].append((date, 'SELL', shares_to_short, current_price))
        
        # Calculate portfolio value
        if shares > 0:  # Long position
            portfolio_value = cash + shares * current_price
        elif shares < 0:  # Short position
            portfolio_value = cash - abs(shares) * current_price
        else:  # Cash position
            portfolio_value = cash
        
        # Store results
        results['dates'].append(date)
        results['positions'].append(position)
        results['shares'].append(shares)
        results['cash'].append(cash)
        results['portfolio_value'].append(portfolio_value)
        
        # Calculate daily return
        if i > 0:
            daily_return = (portfolio_value - results['portfolio_value'][i-1]) / results['portfolio_value'][i-1]
            results['returns'].append(daily_return)
        else:
            results['returns'].append(0)
    
    # Convert to DataFrame for easier analysis
    results_df = pd.DataFrame({
        'position': results['positions'],
        'shares': results['shares'],
        'cash': results['cash'],
        'portfolio_value': results['portfolio_value'],
        'daily_return': results['returns']
    }, index=results['dates'])
    
    return results_df, results['trades']

# Test the backtesting framework
print("Backtesting framework ready...")
print(f"Test period: {len(backtest_data)} trading days")

## 4. Strategy Performance Comparison

In [ ]:
# Backtest both strategies
print("Running backtests...")

# Simple strategy backtest
simple_results, simple_trades = backtest_strategy(backtest_data, backtest_data['simple_signal'])
print(f"\nSimple Strategy: {len(simple_trades)} trades executed")

# Optimal strategy backtest  
optimal_results, optimal_trades = backtest_strategy(backtest_data, backtest_data['optimal_signal'])
print(f"Optimal Strategy: {len(optimal_trades)} trades executed")

# Buy and hold benchmark
buy_hold_signals = pd.Series([1] + [0]*(len(backtest_data)-1), index=backtest_data.index)
buyhold_results, buyhold_trades = backtest_strategy(backtest_data, buy_hold_signals)
print(f"Buy & Hold: {len(buyhold_trades)} trades executed")

In [ ]:
def calculate_metrics(results_df, strategy_name, initial_capital=100000):
    """
    Calculate comprehensive performance metrics
    """
    
    returns = results_df['daily_return']
    final_value = results_df['portfolio_value'].iloc[-1]
    
    # Basic metrics
    total_return = (final_value - initial_capital) / initial_capital
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    
    # Risk metrics
    sharpe_ratio = annualized_return / volatility if volatility > 0 else 0
    max_drawdown = ((results_df['portfolio_value'].cummax() - results_df['portfolio_value']) / results_df['portfolio_value'].cummax()).max()
    
    # Win rate
    winning_days = (returns > 0).sum()
    total_days = len(returns)
    win_rate = winning_days / total_days if total_days > 0 else 0
    
    return {
        'Strategy': strategy_name,
        'Final Value': f'${final_value:,.0f}',
        'Total Return': f'{total_return:.2%}',
        'Annualized Return': f'{annualized_return:.2%}',
        'Volatility': f'{volatility:.2%}',
        'Sharpe Ratio': f'{sharpe_ratio:.2f}',
        'Max Drawdown': f'{max_drawdown:.2%}',
        'Win Rate': f'{win_rate:.2%}'
    }

# Calculate metrics for all strategies
simple_metrics = calculate_metrics(simple_results, 'Simple If/Elif/Else')
optimal_metrics = calculate_metrics(optimal_results, 'Optimal Multi-Signal')
buyhold_metrics = calculate_metrics(buyhold_results, 'Buy & Hold')

# Create comparison table
comparison_df = pd.DataFrame([simple_metrics, optimal_metrics, buyhold_metrics])
print("\n" + "="*80)
print("STRATEGY PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("\n" + "="*80)

## 5. Visualization and Analysis

In [ ]:
# Portfolio value comparison
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(simple_results.index, simple_results['portfolio_value'], label='Simple Strategy', linewidth=2)
plt.plot(optimal_results.index, optimal_results['portfolio_value'], label='Optimal Strategy', linewidth=2)
plt.plot(buyhold_results.index, buyhold_results['portfolio_value'], label='Buy & Hold', linewidth=2)
plt.title('Portfolio Value Over Time')
plt.ylabel('Portfolio Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)

# GLD price vs predictions
plt.subplot(2, 2, 2)
plt.plot(backtest_data.index, backtest_data['actual_price'], label='Actual GLD Price', color='blue', linewidth=2)
plt.plot(backtest_data.index, backtest_data['predicted_price'], label='ARIMA/LSTM Predictions', color='red', linewidth=2, alpha=0.7)
plt.title('GLD Price: Actual vs Predictions')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

# Signal comparison
plt.subplot(2, 2, 3)
signal_dates = backtest_data.index
buy_simple = signal_dates[backtest_data['simple_signal'] == 1]
sell_simple = signal_dates[backtest_data['simple_signal'] == -1]
buy_optimal = signal_dates[backtest_data['optimal_signal'] == 1]
sell_optimal = signal_dates[backtest_data['optimal_signal'] == -1]

plt.plot(backtest_data.index, backtest_data['actual_price'], color='gray', alpha=0.5, linewidth=1)
plt.scatter(buy_simple, backtest_data.loc[buy_simple, 'actual_price'], color='green', marker='^', s=50, label='Simple Buy', alpha=0.7)
plt.scatter(sell_simple, backtest_data.loc[sell_simple, 'actual_price'], color='red', marker='v', s=50, label='Simple Sell', alpha=0.7)
plt.scatter(buy_optimal, backtest_data.loc[buy_optimal, 'actual_price'], color='darkgreen', marker='^', s=30, label='Optimal Buy')
plt.scatter(sell_optimal, backtest_data.loc[sell_optimal, 'actual_price'], color='darkred', marker='v', s=30, label='Optimal Sell')
plt.title('Trading Signals on GLD Price')
plt.ylabel('GLD Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

# Daily returns distribution
plt.subplot(2, 2, 4)
plt.hist(simple_results['daily_return'], bins=30, alpha=0.7, label='Simple Strategy', density=True)
plt.hist(optimal_results['daily_return'], bins=30, alpha=0.7, label='Optimal Strategy', density=True)
plt.hist(buyhold_results['daily_return'], bins=30, alpha=0.7, label='Buy & Hold', density=True)
plt.title('Daily Returns Distribution')
plt.xlabel('Daily Return')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Trade Analysis

In [ ]:
print("\n" + "="*60)
print("TRADE ANALYSIS")
print("="*60)

def analyze_trades(trades, strategy_name):
    if len(trades) == 0:
        print(f"\n{strategy_name}: No trades executed")
        return
    
    trade_df = pd.DataFrame(trades, columns=['Date', 'Action', 'Shares', 'Price'])
    
    print(f"\n{strategy_name}:")
    print(f"  Total trades: {len(trades)}")
    print(f"  Buy trades: {(trade_df['Action'] == 'BUY').sum()}")
    print(f"  Sell trades: {(trade_df['Action'] == 'SELL').sum()}")
    
    if len(trades) > 0:
        print(f"  Average trade size: {trade_df['Shares'].mean():.0f} shares")
        print(f"  Price range: ${trade_df['Price'].min():.2f} - ${trade_df['Price'].max():.2f}")
        
        print("\n  Recent trades:")
        print(trade_df.tail().to_string(index=False))

analyze_trades(simple_trades, "Simple Strategy")
analyze_trades(optimal_trades, "Optimal Strategy")
analyze_trades(buyhold_trades, "Buy & Hold")

## 7. Prediction Accuracy Analysis

In [ ]:
# Analyze how good the ARIMA/LSTM predictions were
print("\n" + "="*60)
print("PREDICTION ACCURACY ANALYSIS")
print("="*60)

# Prediction errors
prediction_error = backtest_data['predicted_price'] - backtest_data['actual_price']
mape = np.mean(np.abs(prediction_error) / backtest_data['actual_price']) * 100
rmse = np.sqrt(np.mean(prediction_error**2))

print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Root Mean Square Error (RMSE): ${rmse:.2f}")
print(f"Prediction bias (mean error): ${prediction_error.mean():.2f}")

# Direction accuracy
actual_direction = (backtest_data['actual_price'].pct_change() > 0).astype(int)
predicted_direction = (backtest_data['predicted_price'].pct_change() > 0).astype(int)
direction_accuracy = (actual_direction == predicted_direction).mean()

print(f"Direction accuracy: {direction_accuracy:.2%}")

# Plot prediction accuracy
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.scatter(backtest_data['actual_price'], backtest_data['predicted_price'], alpha=0.6)
plt.plot([backtest_data['actual_price'].min(), backtest_data['actual_price'].max()], 
         [backtest_data['actual_price'].min(), backtest_data['actual_price'].max()], 'r--', linewidth=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Prediction vs Actual Scatter Plot')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(backtest_data.index, prediction_error, linewidth=1)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
plt.fill_between(backtest_data.index, prediction_error, 0, alpha=0.3)
plt.xlabel('Date')
plt.ylabel('Prediction Error ($)')
plt.title('Prediction Error Over Time')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Key Findings and Recommendations

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS AND RECOMMENDATIONS")
print("="*80)

# Extract key metrics for comparison
simple_return = float(simple_metrics['Total Return'].strip('%')) / 100
optimal_return = float(optimal_metrics['Total Return'].strip('%')) / 100
buyhold_return = float(buyhold_metrics['Total Return'].strip('%')) / 100

simple_sharpe = float(simple_metrics['Sharpe Ratio'])
optimal_sharpe = float(optimal_metrics['Sharpe Ratio'])
buyhold_sharpe = float(buyhold_metrics['Sharpe Ratio'])

print("\n1. STRATEGY PERFORMANCE RANKING:")
strategies = [('Simple If/Elif/Else', simple_return, simple_sharpe),
              ('Optimal Multi-Signal', optimal_return, optimal_sharpe),
              ('Buy & Hold', buyhold_return, buyhold_sharpe)]

strategies.sort(key=lambda x: x[1], reverse=True)  # Sort by return
for i, (name, ret, sharpe) in enumerate(strategies, 1):
    print(f"   {i}. {name}: {ret:.2%} return, {sharpe:.2f} Sharpe")

print("\n2. PREDICTION MODEL EFFECTIVENESS:")
print(f"   - MAPE: {mape:.2f}% (Lower is better)")
print(f"   - Direction Accuracy: {direction_accuracy:.2%} (Random = 50%)")
print(f"   - RMSE: ${rmse:.2f}")

if direction_accuracy > 0.55:
    print("   ✅ Model shows predictive skill above random")
else:
    print("   ⚠️  Model accuracy is close to random - consider improvements")

print("\n3. TRADING FREQUENCY:")
print(f"   - Simple Strategy: {len(simple_trades)} trades")
print(f"   - Optimal Strategy: {len(optimal_trades)} trades")
print(f"   - Buy & Hold: {len(buyhold_trades)} trades")

print("\n4. RECOMMENDATIONS:")

if optimal_return > simple_return and optimal_return > buyhold_return:
    print("   ✅ OPTIMAL STRATEGY WINS: Use the multi-signal approach")
    print("   - Combines prediction confidence with technical indicators")
    print("   - Better risk management through multiple signals")
elif simple_return > buyhold_return:
    print("   ✅ SIMPLE STRATEGY EFFECTIVE: Basic thresholds work well")
    print("   - If/elif/else is surprisingly competitive")
    print("   - Consider this for simplicity and interpretability")
else:
    print("   ⚠️  BUY & HOLD DOMINATES: Active strategies underperform")
    print("   - Consider improving prediction model or strategy logic")
    print("   - Transaction costs may be too high for active trading")

print("\n5. FUTURE IMPROVEMENTS:")
print("   - Add stop-loss and take-profit levels")
print("   - Implement position sizing based on confidence")
print("   - Consider longer prediction horizons")
print("   - Add more technical indicators for confirmation")
print("   - Test different threshold values")

print("\n" + "="*80)